# 05_02 Sensitive-Fraction Sweep

This sweep asks how the composition of the item bank changes abandonment and information quality.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "_sweep_helpers.py").exists():
    sys.path.append(str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "notebooks" / "_sweep_helpers.py").exists():
    sys.path.append(str(NOTEBOOK_DIR / "notebooks"))
else:
    sys.path.append(str(Path(r"C:/Users/49160/Adaptive-Onboarding/notebooks")))

from _sweep_helpers import (
    POLICY_STYLE,
    config_label,
    equalize_y_axes,
    load_run,
    load_sweep,
    plot_dimension_group_lines,
    plot_metric_lines,
    plot_pareto,
    plot_weighted_delta,
    save,
    setup,
    show_sweeps,
    style_ax,
)

setup()

## Select Sweep

In [ ]:
PARAM = "sensitive_frac"
SWEEP_FILE = None  # e.g. "20260428_164858_sweep_sensitive_frac.json"
CONFIG_FILTER = {}  # e.g. {"dim": 6, "horizon": 20, "n_users": 500}

show_sweeps(PARAM)
sweep = load_sweep(PARAM, sweep_file=SWEEP_FILE, config_filter=CONFIG_FILTER)
print(config_label(sweep))

## Main Outcomes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_metric_lines(
    axes[0], sweep, "dropout_rate",
    x_scale=100, y_scale=100,
    xlabel="Sensitive items in bank (%)",
    ylabel="Episode dropout rate (%)",
    title="Dropout rate",
)
plot_metric_lines(
    axes[1], sweep, "mean_final_d_error",
    x_scale=100,
    xlabel="Sensitive items in bank (%)",
    ylabel="Mean D-error",
    title="D-error, all episodes",
)
plot_metric_lines(
    axes[2], sweep, "mean_final_d_error_completed",
    x_scale=100,
    xlabel="Sensitive items in bank (%)",
    ylabel="Mean D-error",
    title="D-error, completed only",
)
equalize_y_axes(axes[1], axes[2])
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_02_sensitive_fraction_main")
plt.show()

## Trait-Group Uncertainty

For axis-targeted sensitivity configs, this splits posterior marginal variance into sensitive versus other traits as the sensitive item fraction changes.

In [ ]:
metric = "mean_final_d_error_by_dimension"
comparison_policies = [p for p in ["surrogate_unweighted", "surrogate_weighted"] if p in sweep["conditions"][0]["policies"]]

if metric not in next(iter(sweep["conditions"][-1]["policies"].values())):
    print("This sweep was generated before per-dimension D-error was saved. Re-run experiments.sweep to populate it.")
elif not sweep.get("fixed_config", {}).get("sensitive_axes"):
    print("No sensitive_axes configured for this sweep; this grouped trait view is most useful for axis-targeted sensitivity.")
else:
    fig, axes = plt.subplots(1, len(comparison_policies), figsize=(5.8 * len(comparison_policies), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, policy in zip(axes, comparison_policies):
        plot_dimension_group_lines(
            ax,
            sweep,
            metric,
            policy=policy,
            x_scale=100,
            xlabel="Sensitive items in bank (%)",
            ylabel="Mean posterior variance" if ax is axes[0] else "",
            title=POLICY_STYLE[policy]["label"],
        )
    fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
    plt.tight_layout()
    # save(fig, "fig_05_02_trait_group_uncertainty")
    plt.show()

## Exposure Mechanism

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_metric_lines(
    axes[0], sweep, "sensitive_rate",
    x_scale=100, y_scale=100,
    xlabel="Sensitive items in bank (%)",
    ylabel="Sensitive questions / questions asked (%)",
    title="Sensitive exposure rate",
)
plot_metric_lines(
    axes[1], sweep, "mean_n_answered",
    x_scale=100,
    xlabel="Sensitive items in bank (%)",
    ylabel="Mean questions answered",
    title="Questions answered",
)
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_02_sensitive_fraction_exposure")
plt.show()

## Dropout-Risk vs Error and Weighted Delta

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_pareto(
    axes[0], sweep,
    y_metric="mean_estimation_error",
    ylabel="Mean estimation error (lower is better)",
    title="Dropout-risk vs estimation error",
)
plot_weighted_delta(
    axes[1], sweep, "dropout_rate",
    x_scale=100, y_scale=100,
    xlabel="Sensitive items in bank (%)",
    ylabel="Percentage-point delta",
    title="Delta dropout rate",
)
plot_weighted_delta(
    axes[2], sweep, "mean_estimation_error",
    x_scale=100,
    xlabel="Sensitive items in bank (%)",
    ylabel="Weighted - unweighted",
    title="Delta estimation error",
)
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_02_sensitive_fraction_tradeoff")
plt.show()

## Notes

- 
